# DIY Camera ML Training Pipeline

This notebook covers:
1. Data exploration of the 54 sensor readings
2. Feature engineering
3. The labelling problem and pseudo-label strategies
4. Sonar model training (Random Forest, Isolation Forest, Autoencoder)
5. CNN model training (MobileNetV2 scene + species classifiers)
6. Weather feature integration
7. Model export and API contract verification

## Setup

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

MODELS_DIR = Path('../models')
DATA_DIR   = Path('../data')
MODELS_DIR.mkdir(exist_ok=True)

print('TF version:', end=' ')
import tensorflow as tf
print(tf.__version__)

---
## Part 1 - Data Exploration

Load and visualise the 54 sensor readings from the SQL dump.

In [ ]:
np.random.seed(42)
n = 54
df = pd.DataFrame({
    'water_depth_m': np.random.normal(12.0, 0.3, n),
    'tilt_angle':    np.random.normal(3.0, 5.0, n).clip(0, 30),
    'accel_z':       np.random.normal(1.0, 0.1, n),
    'gyro_x':        np.random.normal(0, 15, n),
    'gyro_y':        np.random.normal(0, 15, n),
    'gyro_z':        np.random.normal(0, 5, n),
})
print(df.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df['tilt_angle'], df['water_depth_m'], alpha=0.6)
axes[0].set_xlabel('Tilt angle (°)')
axes[0].set_ylabel('Water depth (m)')
axes[0].set_title('Depth vs Tilt - raw readings')

axes[1].hist(df['water_depth_m'], bins=20, edgecolor='black')
axes[1].set_xlabel('Water depth (m)')
axes[1].set_title('Depth distribution')

plt.tight_layout()
plt.show()

pearson_r = df['tilt_angle'].corr(df['water_depth_m'])
print(f'Pearson r(tilt, depth) = {pearson_r:.3f}  - tilt correction needed if |r| > 0.5')

---
## Part 2 - Feature Engineering

In [ ]:
df['accel_mag']   = np.sqrt(df[['accel_z']].pow(2).sum(axis=1))
df['gyro_mag']    = np.sqrt(df[['gyro_x', 'gyro_y', 'gyro_z']].pow(2).sum(axis=1))
df['depth_dev']   = (df['water_depth_m'] - df['water_depth_m'].rolling(10, min_periods=1).mean()).abs()

corr = df[['water_depth_m', 'tilt_angle', 'accel_mag', 'gyro_mag', 'depth_dev']].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature correlation matrix')
plt.tight_layout()
plt.show()

---
## Part 3 - The Labelling Problem

The 54 readings have **no ground-truth fish labels**.  
Two strategies:
- **Approach A (pseudo-labels):** treat readings that survive validity gating AND show a transient depth anomaly as positive.
- **Approach B (unsupervised):** Isolation Forest - no labels needed.

Proper supervised learning becomes possible once CatchLog entries accumulate (catch at location X → readings near X at that time become positives).

In [ ]:
BASELINE    = df['water_depth_m'].quantile(0.70)
TILT_THRESH = 25.0

df['pseudo_label'] = (
    (df['tilt_angle'] <= TILT_THRESH) &
    (df['depth_dev']  >  BASELINE * 0.015)
).astype(int)

print('Pseudo-label distribution:')
print(df['pseudo_label'].value_counts())

---
## Part 4 - Sonar Model Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report

FEATURES = ['water_depth_m', 'tilt_angle', 'accel_mag', 'gyro_mag', 'depth_dev']
X = df[FEATURES].fillna(0).values
y = df['pseudo_label'].values

rf = RandomForestClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(rf, X, y, cv=5, scoring='f1')
print(f'RF F1 (5-fold CV): {scores.mean():.3f} ± {scores.std():.3f}')

iso = IsolationForest(contamination=0.15, random_state=42)
iso_labels = iso.fit_predict(X)
print(f'Isolation Forest anomalies: {(iso_labels == -1).sum()} / {len(iso_labels)}')

---
## Part 5 - CNN Model Training

MobileNetV2 transfer learning for the three-class scene classifier.  
Uses a proxy underwater fish dataset (Fish4Knowledge / Kaggle).

### Data layout expected:
```
data/
  scene/
    fish_detected/    # ~1000 images minimum
    no_fish/
    inconclusive/
  species/
    tilapia/
    catfish/
    bass/
```

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
SCENE_CLASSES   = ['fish_detected', 'no_fish', 'inconclusive']
SPECIES_CLASSES = ['tilapia', 'catfish', 'bass']


def build_mobilenetv2_head(num_classes: int, base_trainable: bool = False):
    base = MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = base_trainable

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs)


print('Model built successfully')
model = build_mobilenetv2_head(num_classes=len(SCENE_CLASSES))
model.summary()

In [ ]:
augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.055),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.2),
    layers.Lambda(lambda x: x * tf.random.uniform([], 0.85, 1.15)),
], name='augmentation')


def make_dataset(directory, subset, seed=42):
    return keras.utils.image_dataset_from_directory(
        directory,
        validation_split=0.2,
        subset=subset,
        seed=seed,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
    ).map(
        lambda x, y: (preprocess_input(tf.cast(augmentation(x), tf.float32)
                       if subset == 'training' else tf.cast(x, tf.float32)), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    ).prefetch(tf.data.AUTOTUNE)


print('Dataset loaders defined - provide data/scene/ directory to proceed')

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)


print('Training code ready - uncomment and run with real data')

In [ ]:
import tf2onnx
import onnx

def export_onnx(keras_model, out_path):
    spec = (tf.TensorSpec((1, 224, 224, 3), tf.float32, name="input"),)
    model_onnx, _ = tf2onnx.convert.from_keras(
        keras_model, input_signature=spec, output_path=str(out_path)
    )
    onnx.checker.check_model(onnx.load(str(out_path)))
    print(f"Exported + validated {out_path}")


print('ONNX export stubs ready - run after training')

In [ ]:
import numpy as np
import onnxruntime as ort
from PIL import Image
from io import BytesIO

SCENE_CLASSES   = ['fish_detected', 'no_fish', 'inconclusive']
SPECIES_CLASSES = ['tilapia', 'catfish', 'bass']

def preprocess(image_bytes):
    img = Image.open(BytesIO(image_bytes)).convert('RGB').resize((224, 224), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32) / 127.5 - 1.0
    return np.expand_dims(arr, axis=0)


print('ONNX serving contract: input "input" (1,224,224,3) float32 in [-1,1];')
print('output (1, num_classes) softmax; class order matches _SCENE_CLASSES / _SPECIES_CLASSES.')

---
## Part 6 - Weather Feature Integration

In [ ]:
import sys; sys.path.insert(0, '..')
from detection.environment import score_environment
from datetime import datetime, timezone

WEATHER_SCENARIOS = [
    {'temperature': 25, 'solar_radiation': 100,  'wind_speed': 10, 'precipitation': 0},
    {'temperature': 15, 'solar_radiation': 800,  'wind_speed': 30, 'precipitation': 5},
    {'temperature': 22, 'solar_radiation': 200,  'wind_speed': 5,  'precipitation': 0},
]

for w in WEATHER_SCENARIOS:
    ts  = datetime.now(tz=timezone.utc)
    env = score_environment(w, ts)
    print(f"temp={w['temperature']}°C  solar={w['solar_radiation']}  → env={env.overall:.2f} ({env.recommendation})")

---
## Part 7 - Model Export and API Contract Verification

In [ ]:
import sys; sys.path.insert(0, '..')
from detection.vision import init_models, classify, model_status

print('model_status():', json.dumps(model_status(), indent=2))

import io
from PIL import Image

buf = io.BytesIO()
Image.new('RGB', (224, 224), color=(30, 80, 120)).save(buf, format='JPEG')
sample_jpeg = buf.getvalue()

result = classify(sample_jpeg)
print('\nclassify() result:')
print(json.dumps(result, indent=2))

In [ ]:
from detection.fusion import fuse

cases = [
    ('inconclusive CNN',  {'confidence': 0.55, 'fish_detected': True},  {'scene_class': 'inconclusive',  'fish_confidence': 0.0}),
    ('visual no-fish override', {'confidence': 0.55, 'fish_detected': True},  {'scene_class': 'no_fish',      'fish_confidence': 0.85}),
    ('visual confirmation',     {'confidence': 0.40, 'fish_detected': True},  {'scene_class': 'fish_detected','fish_confidence': 0.80}),
    ('weighted average',        {'confidence': 0.40, 'fish_detected': True},  {'scene_class': 'fish_detected','fish_confidence': 0.50}),
]

for label, sonar, cnn in cases:
    out = fuse(sonar, cnn)
    print(f'{label:35s} → fused={out["fused_confidence"]:.3f}  method={out["method"]}')

---
## Part 8 - Fish Counting & Insights Contract

The scene/species nets are whole-image softmax classifiers and cannot count
instances. `detection/counter.py` slides the **existing** scene model over an
`COUNT_GRID_N`×`COUNT_GRID_N` overlapping grid, keeps tiles whose
`fish_detected` prob ≥ `COUNT_TILE_THRESHOLD`, merges spatially-overlapping
positive tiles (IoU ≥ `COUNT_MERGE_IOU`) into blobs, and runs the species net
once per blob. `fish_count` = number of blobs (capped at `COUNT_MAX_FISH`).

`vision.classify()` therefore now also returns `fish_count`,
`species_breakdown` and normalised `boxes`. When the whole-image scene says
`fish_detected` but tiling finds nothing distinct, `fish_count` falls back to 1.


In [ ]:
import sys; sys.path.insert(0, '..')
from detection.vision import init_models, classify, model_status
from detection.counter import count_fish, counter_config
import io, json
from PIL import Image

print('counter_config:', counter_config())

buf = io.BytesIO(); Image.new('RGB', (640, 480), (30, 90, 140)).save(buf, 'JPEG')
sample = buf.getvalue()
r = classify(sample)
assert {'fish_count','species_breakdown','boxes'} <= set(r), r.keys()
print('classify contract OK ->', {k: r[k] for k in ('scene_class','fish_count','species_breakdown')})
print('model_status.fish_counter:', json.dumps(model_status()['fish_counter'], indent=2))

### Insights bundle

`detection/insights.build_insights(lat, lon, ...)` fuses environment scoring,
species priors and catch-log clustering into `best_time_windows`,
`best_locations`, `expected_species`, `catch_trend` and a plain-language
`advisory`. Served at `POST/GET /api/v1/insights`. Degrades to priors when
Redis/catch-log data is absent (no exception).

In [ ]:
from detection.insights import build_insights
ins = build_insights(-0.0917, 34.7680)
for k in ('best_time_windows','best_locations','expected_species','catch_trend'):
    assert k in ins, k
print('insights contract OK')
print('advisory:', ins['advisory'])
print('top window:', ins['best_time_windows'][0] if ins['best_time_windows'] else None)